## Setup

In [1]:
from jaxcmr.datasets import load_data

model_name = 'Base_CMR'
data_tag = 'LohnasKahana2014'
param_path = 'D:/data/base_cmr_parameters.json'
data_path = 'D:/data/{}.h5'
    
data_path = data_path.format(data_tag)
data = load_data(data_path)

presentations = data['pres_itemnos']
trials = data['recalls']
list_types = data['list_type'].flatten()
subjects = data['subject'].flatten()

presentations[:3]

array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 12, 13, 14, 15,
        16, 17, 10, 18, 19, 20, 19, 21, 22, 23, 20, 24, 25, 26, 22, 27,
        28, 24, 29, 30, 31, 32, 33, 34],
       [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32,
        33, 34, 35, 36, 37, 38, 39, 40],
       [ 1,  2,  3,  4,  5,  6,  5,  7,  2,  1,  8,  3,  4,  8,  6,  7,
         9, 10, 11, 12, 11, 13, 14, 15, 10,  9, 16, 13, 12, 14, 16, 17,
        15, 18, 19, 20, 17, 20, 19, 18]])

## Numba Implementation

In [2]:
from compmempy.evaluation import variable_presentations_data_likelihood
from compmempy.models.memorysearch import Base_CMR
from compmempy.parameters import Parameters

full_parameters = Parameters(param_path)
parameters = full_parameters.fixed

In [3]:
presentations[subjects==1].shape

(48, 40)

In [4]:
# variable_presentations_data_likelihood(
#     trials[subjects==1][:1], presentations[subjects==1][:1], Base_CMR, parameters, False)

> 124.4956968964472

### Step Through

In [5]:
import numpy as np

trial = trials[subjects==1][0]
presentation = presentations[subjects==1][0]

item_count = np.max(presentation)
items = np.eye(item_count)
likelihood = np.ones((1, 40))
trial_index = 0
ignore_first_recall = False
lb = np.finfo(float).eps

trial, presentation

(array([ 1,  2,  3,  4,  5,  6,  7,  9, 10, 11, 17, 14, 12, 15, 25, 20, 28,
        30, 39, 38, 37, 18,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0]),
 array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 12, 13, 14, 15, 16,
        17, 10, 18, 19, 20, 19, 21, 22, 23, 20, 24, 25, 26, 22, 27, 28, 24,
        29, 30, 31, 32, 33, 34]))

In [6]:
model = Base_CMR(items, item_count, parameters)
model.experience(model.items[presentation-1])

model.start_retrieving()
recall_list = []
for recall_index in range(min(len(trial) + 1, item_count)):

    # identify index of item recalled; if zero then recall is over
    if recall_index == len(trial) and len(trial) < item_count:
        recall = 0
    elif trial[recall_index] == 0:
        recall = 0
    else:
        recall = presentation[trial[recall_index]-1]

    recall_list.append(recall)

    # store probability of and simulate recalling item with this index
    if (not ignore_first_recall) or (ignore_first_recall and recall_index > 0):
        print(model.outcome_probabilities())
        likelihood[trial_index, recall_index] = \
            model.outcome_probabilities()[recall]
        if likelihood[trial_index, recall_index] <= 0:
            #print('Likelihood is not greater than zero', trial_index, recall_index, recall, trial, model.outcome_probabilities())
            likelihood[trial_index, recall_index] = lb

    if recall == 0 or recall_index+1 == item_count:
        break
    model.retrieve(recall)

    break

# reset model to its pre-retrieval (but post-encoding) state
# model.retrieve(0)

[0.003449   0.70466359 0.09872473 0.0195591  0.00740708 0.00451317
 0.0033263  0.00268435 0.00230912 0.00208609 0.00199487 0.00187625
 0.00187808 0.00178463 0.00177346 0.00176773 0.00176587 0.00176739
 0.0017932  0.001984   0.00242358 0.00192631 0.00357327 0.00218187
 0.00806951 0.00312411 0.00291912 0.00351169 0.00442535 0.00465635
 0.00660934 0.0098762  0.01534086 0.02448188 0.03977255]


In [7]:
recall_list

[1]

In [8]:
likelihood

array([[0.70466359, 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ,
        1.        , 1.        , 1.        , 1.        , 1.        ]])

In [9]:
-np.sum(np.log(likelihood))

0.3500347676502117

## Jax Implementation

In [10]:
from jaxcmr.memorysearch import (
    variable_presentations_data_likelihood, trial_item_count, predict_and_simulate_pres_and_trial, recall_by_item_index, log_likelihood)

from jaxcmr.memorysearch import BaseCMR
from jax import numpy as jnp, vmap
import json

with open(param_path) as f:
    full_parameters = json.load(f)
parameters = full_parameters['fixed']

jax_presentations = jnp.array(presentations[subjects==1])
jax_trials = jnp.array(trials[subjects==1])
jax_trials = vmap(recall_by_item_index)(jax_presentations, jax_trials) # TODO: scary important

item_counts = vmap(trial_item_count)(jax_presentations)

jax_trials[0], jax_presentations[0]

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


(Array([ 1,  2,  3,  4,  5,  6,  7,  9, 10, 11, 16, 13, 12, 14, 22, 18, 24,
        26, 33, 32, 31, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0], dtype=int32),
 Array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 12, 13, 14, 15, 16,
        17, 10, 18, 19, 20, 19, 21, 22, 23, 20, 24, 25, 26, 22, 27, 28, 24,
        29, 30, 31, 32, 33, 34], dtype=int32))

### Predict and Simulate Trial

In [12]:
from jaxcmr.memorysearch import predict_and_simulate_trial, start_retrieving, experience, outcome_probabilities

item_count = 34
model_init = BaseCMR.create
presentation = jax_presentations[item_counts == item_count][0]
trial = jax_trials[item_counts == item_count][0]

trial, presentation

(Array([ 1,  2,  3,  4,  5,  6,  7,  9, 10, 11, 16, 13, 12, 14, 22, 18, 24,
        26, 33, 32, 31, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0], dtype=int32),
 Array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 12, 13, 14, 15, 16,
        17, 10, 18, 19, 20, 19, 21, 22, 23, 20, 24, 25, 26, 22, 27, 28, 24,
        29, 30, 31, 32, 33, 34], dtype=int32))

In [13]:
model = model_init(item_count, presentation.shape[0], parameters)
model = start_retrieving(experience(model, presentation))
print(outcome_probabilities(model))
model, likelihood = predict_and_simulate_trial(model, trial[:1])

model, likelihood

[0.003449   0.62371194 0.08738331 0.01731222 0.00655623 0.00399477
 0.00294425 0.00237605 0.00204392 0.00184652 0.00176578 0.00166078
 0.0016624  0.00157969 0.0015698  0.00156473 0.00156309 0.00156443
 0.00158727 0.00175616 0.00214524 0.00170509 0.00316286 0.0019313
 0.00714258 0.00276529 0.00258385 0.00310835 0.00391705 0.00788371
 0.01205206 0.0190614  0.03080826 0.05047098 0.0833697 ]


(<jaxcmr.memorysearch.BaseCMR.BaseCMR at 0x204ce1c9fd0>,
 Array([0.62371194], dtype=float32, weak_type=True))

In [15]:
from jaxcmr.memorysearch import log_likelihood

log_likelihood(likelihood)

Array(0.47206664, dtype=float32)

### Predict and Simulate Pres and Trial

In [19]:
model_create_fn = BaseCMR.create
presentation = jax_presentations[item_counts == item_count][0]
trial = jax_trials[item_counts == item_count][0]

presentation, trial

(Array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 12, 13, 14, 15, 16,
        17, 10, 18, 19, 20, 19, 21, 22, 23, 20, 24, 25, 26, 22, 27, 28, 24,
        29, 30, 31, 32, 33, 34], dtype=int32),
 Array([ 1,  2,  3,  4,  5,  6,  7,  9, 10, 11, 16, 13, 12, 14, 22, 18, 24,
        26, 33, 32, 31, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0], dtype=int32))

In [20]:
model, likelihoods = predict_and_simulate_pres_and_trial(
        model_create_fn,
        item_count,
        presentation,
        trial,
        parameters,
    )


model, likelihoods

(<jaxcmr.memorysearch.BaseCMR.BaseCMR at 0x2b989c38c50>,
 Array([6.2371194e-01, 6.6461331e-01, 4.4454554e-01, 3.1295550e-01,
        2.7370840e-01, 2.6364917e-01, 2.5940681e-01, 1.5693089e-01,
        2.1598317e-01, 1.3948157e-01, 2.8556678e-02, 4.8909672e-02,
        1.1182680e-01, 1.4196642e-01, 7.5592585e-03, 6.3565146e-08,
        1.6037637e-07, 1.6693079e-07, 4.5407518e-08, 1.2073659e-07,
        1.6857074e-07, 4.0466951e-08, 9.9999857e-01, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00],      dtype=float32, weak_type=True))

In [21]:
trial[:(trial!=0).sum()]

Array([ 1,  2,  3,  4,  5,  6,  7,  9, 10, 11, 16, 13, 12, 14, 22, 18, 24,
       26, 33, 32, 31, 17], dtype=int32)

In [22]:
likelihoods[:(trial!=0).sum()+1]

Array([6.2371194e-01, 6.6461331e-01, 4.4454554e-01, 3.1295550e-01,
       2.7370840e-01, 2.6364917e-01, 2.5940681e-01, 1.5693089e-01,
       2.1598317e-01, 1.3948157e-01, 2.8556678e-02, 4.8909672e-02,
       1.1182680e-01, 1.4196642e-01, 7.5592585e-03, 6.3565146e-08,
       1.6037637e-07, 1.6693079e-07, 4.5407518e-08, 1.2073659e-07,
       1.6857074e-07, 4.0466951e-08, 9.9999857e-01],      dtype=float32, weak_type=True)

In [23]:
log_likelihood(likelihoods)

Array(141.06567, dtype=float32)

### Multiple Trials

In [25]:
model_create_fn = BaseCMR.create

likelihood_fn = variable_presentations_data_likelihood(
    model_create_fn,
    jax_presentations[:1],
    jax_trials[:1]
    )

likelihood_fn(parameters)

Array(141.06567, dtype=float32)